In [4]:
# tune_optuna_bilstm_wind.py
import argparse, json, math, random
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import matplotlib.pyplot as plt

# ======= Columns =======
TIME_COL   = "TIMESTAMP"
TARGET_COL = "TARGETVAR"
BASE_FEATS = ["U10", "V10", "U100", "V100"]

# ======= Fixed FE =======
LAGS_Y     = [1, 3, 6, 12, 24]
LAGS_SPEED = [1, 3, 6]
ROLLS_Y    = [6, 12, 24]

# ======= Splits =======
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15  # Test = rest (≈ 0.15)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======= Repro =======
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

# ======= FE =======
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create new features out of the existing data.
    """
    out = df.copy()
    # wind speed & direction
    out["speed10"]  = np.sqrt(out["U10"]**2  + out["V10"]**2)
    out["speed100"] = np.sqrt(out["U100"]**2 + out["V100"]**2)
    dir10  = np.arctan2(out["V10"],  out["U10"])
    dir100 = np.arctan2(out["V100"], out["U100"])
    out["dir10_sin"], out["dir10_cos"]   = np.sin(dir10),  np.cos(dir10)
    out["dir100_sin"], out["dir100_cos"] = np.sin(dir100), np.cos(dir100)

    # shear & veer
    out["shear_speed"] = out["speed100"] - out["speed10"]
    veer = dir100 - dir10
    out["veer_sin"], out["veer_cos"] = np.sin(veer), np.cos(veer)

    # time features
    out["hour"] = out[TIME_COL].dt.hour
    out["day"]  = out[TIME_COL].dt.dayofyear
    out["hour_sin"] = np.sin(2*np.pi*out["hour"]/24.0)
    out["hour_cos"] = np.cos(2*np.pi*out["hour"]/24.0)
    out["day_sin"]  = np.sin(2*np.pi*out["day"]/366.0)
    out["day_cos"]  = np.cos(2*np.pi*out["day"]/366.0)

    # target lags (shifted)
    for L in LAGS_Y:
        out[f"y_lag{L}"] = out[TARGET_COL].shift(L)

    # rolling means of y (shift to avoid leakage)
    for W in ROLLS_Y:
        out[f"y_roll{W}"] = (
            out[TARGET_COL].shift(1).rolling(W, min_periods=W).mean()
        )

    # speed lags
    for L in LAGS_SPEED:
        out[f"speed10_lag{L}"]  = out["speed10"].shift(L)
        out[f"speed100_lag{L}"] = out["speed100"].shift(L)

    return out


def build_feat_list():
    """
    List of all feature columns used to train the model.
    """
    return (
        BASE_FEATS +
        [
            "speed10", "speed100",
            "dir10_sin", "dir10_cos",
            "dir100_sin", "dir100_cos",
            "shear_speed", "veer_sin", "veer_cos",
            "hour_sin", "hour_cos",
            "day_sin", "day_cos"
        ] +
        [f"y_lag{L}" for L in LAGS_Y] +
        [f"y_roll{W}" for W in ROLLS_Y] +
        [f"speed10_lag{L}"  for L in LAGS_SPEED] +
        [f"speed100_lag{L}" for L in LAGS_SPEED]
    )

# ======= Data I/O =======
def load_data(file_path: str) -> pd.DataFrame:
    "Read the dataset (use CLI args, not hardcoded filename)."
    df = pd.read_excel(file_path)
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], infer_datetime_format=True, errors="coerce")
    df = df.sort_values(TIME_COL).reset_index(drop=True)
    return df

def smape(y_true, y_pred, eps=1e-8):
    "Symmetric mean absolute percentage error."
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return 100.0 * np.mean(
        2.0 * np.abs(y_pred - y_true) /
        (np.abs(y_pred) + np.abs(y_true) + eps)
    )

# ======= Sequences / Dataset =======
def make_sequences(X, y, lookback):
    "Create sequences for supervised learning."
    Xs, ys = [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i-lookback:i, :])
        ys.append(y[i, 0])
    return np.array(Xs, np.float32), np.array(ys, np.float32).reshape(-1, 1)


class SeqDS(Dataset):
    "Simple sequence dataset for PyTorch."
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]

# ======= Model =======
class BiLSTMRegressor(nn.Module):
    """
    BiLSTM with last-step + mean pooling and MLP head.
    """
    def __init__(self, input_size, hidden_size, num_layers, dropout, bidirectional=True):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )
        self.out_size = hidden_size * (2 if bidirectional else 1)
        self.norm = nn.LayerNorm(self.out_size)
        self.head = nn.Sequential(
            nn.Linear(self.out_size * 2, self.out_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.out_size, 1),
        )

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        o, _ = self.lstm(x)          # (B, T, H_out)
        last = self.norm(o[:, -1, :])
        mean = self.norm(o.mean(dim=1))
        h = torch.cat([last, mean], dim=1)
        return self.head(h)

# ======= Training =======
def train_fold(Xtr, ytr, Xva, yva, params, max_epochs, es_patience, log_target, clip_norm):
    """
    Train one fold and return best RMSE, best state_dict, and scalers.
    """
    # scalers fit on train only
    xsc = StandardScaler().fit(Xtr)
    ysc = StandardScaler().fit(ytr)
    Xtr_s, ytr_s = xsc.transform(Xtr), ysc.transform(ytr)
    Xva_s, yva_s = xsc.transform(Xva), ysc.transform(yva)

    # sequences
    lookback = params["lookback"]
    Xtr_seq, ytr_seq = make_sequences(Xtr_s, ytr_s, lookback)
    Xva_seq, yva_seq = make_sequences(Xva_s, yva_s, lookback)

    if len(Xtr_seq) < 16 or len(Xva_seq) < 16:
        return float("inf"), None, None  # degenerate

    tr_loader = DataLoader(SeqDS(Xtr_seq, ytr_seq), batch_size=params["batch"], shuffle=True)
    va_loader = DataLoader(SeqDS(Xva_seq, yva_seq), batch_size=params["batch"], shuffle=False)

    model = BiLSTMRegressor(
        input_size=Xtr_seq.shape[-1],
        hidden_size=params["hidden"],
        num_layers=params["layers"],
        dropout=params["dropout"],
        bidirectional=params["bidir"]
    ).to(DEVICE)

    # Use MSE since we're optimizing RMSE
    loss_fn = nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=params["lr"], weight_decay=params["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt, T_0=10, T_mult=2, eta_min=1e-5
    )

    def inv_target(y_scaled):
        "Inverse-transform predictions/targets from scaled space."
        y = ysc.inverse_transform(y_scaled).ravel()
        if log_target:
            y = np.expm1(y)
        return y

    best = float("inf")
    no_improve = 0
    best_state = None

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            if clip_norm is not None:
                nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
            opt.step()
        sched.step(epoch - 1)

        # Validation RMSE in original units
        model.eval()
        preds_s, trues_s = [], []
        with torch.no_grad():
            for xb, yb in va_loader:
                pr = model(xb.to(DEVICE)).cpu().numpy()
                preds_s.append(pr)
                trues_s.append(yb.numpy())
        preds_s = np.vstack(preds_s)
        trues_s = np.vstack(trues_s)
        y_pred = inv_target(preds_s)
        y_true = inv_target(trues_s)
        rmse = math.sqrt(mean_squared_error(y_true, y_pred))

        if rmse < best - 1e-6:
            best = rmse
            no_improve = 0
            best_state = model.state_dict()
        else:
            no_improve += 1
            if no_improve >= es_patience:
                break

    return best, best_state, (xsc, ysc)


def make_walkforward_indices(n_total, n_test):
    """
    Reserve last n_test for final Test.
    Create 3 CV folds on the preceding region with growing Train, small Val windows.
    """
    n_tv = n_total - n_test
    # folds: [0:0.60->0.75], [0:0.75->0.85], [0:0.85->0.95] of the train+val region
    cut1 = int(n_tv * 0.60); val1 = int(n_tv * 0.75)
    cut2 = int(n_tv * 0.75); val2 = int(n_tv * 0.85)
    cut3 = int(n_tv * 0.85); val3 = int(n_tv * 0.95)

    folds = [
        (0, cut1, cut1, val1),
        (0, cut2, cut2, val2),
        (0, cut3, cut3, val3),
    ]
    return folds, n_tv

# ======= Objective for Optuna =======
def objective(trial, df: pd.DataFrame, max_epochs: int, es_patience: int):
    params = {
        "lookback": trial.suggest_categorical("lookback", [6, 12, 24]),
        "hidden":   trial.suggest_categorical("hidden",   [128, 256]),
        "layers":   trial.suggest_categorical("layers",   [1, 2]),
        "dropout":  trial.suggest_categorical("dropout", [0.1, 0.3]),
        "bidir": trial.suggest_categorical("bidir", [True, False]),
        "batch":    trial.suggest_categorical("batch", [64, 128]),
        "lr": trial.suggest_categorical("lr",[3e-4, 7e-4, 1e-3]),
        "wd": trial.suggest_categorical("weight_decay",[0.0, 1e-5, 3e-5, 1e-4]),
        "clip": trial.suggest_categorical("clip_norm", [0.5, 1.0]),
        "log_target": trial.suggest_categorical("log_target", [False, True]),
    }

    # Feature engineering + target transform
    dfe = add_engineered_features(df)
    if params["log_target"]:
        dfe["_y"] = np.log1p(dfe[TARGET_COL].clip(lower=0)).astype(np.float32)
    else:
        dfe["_y"] = dfe[TARGET_COL].astype(np.float32)
    dfe = dfe.dropna().reset_index(drop=True)

    feat_cols = build_feat_list()
    X_all = dfe[feat_cols].to_numpy(np.float32)
    y_all = dfe[["_y"]].to_numpy(np.float32)

    n_total = len(dfe)
    n_test  = int(len(dfe) * (1 - (TRAIN_RATIO + VAL_RATIO)))  # ≈ last 15% kept for final test
    folds, n_tv = make_walkforward_indices(n_total, n_test)
    # Use only the last (most recent) fold for tuning to reduce time
    folds = [folds[-1]]


    fold_rmses = []
    for (tr_start, tr_end, va_start, va_end) in folds:
        Xtr = X_all[tr_start:tr_end]; ytr = y_all[tr_start:tr_end]
        Xva = X_all[va_start:va_end]; yva = y_all[va_start:va_end]
        rmse, _, _ = train_fold(
            Xtr, ytr, Xva, yva,
            params=params,
            max_epochs=max_epochs,
            es_patience=es_patience,
            log_target=params["log_target"],
            clip_norm=params["clip"]
        )
        fold_rmses.append(rmse)

    avg_rmse = float(np.mean(fold_rmses))
    trial.set_user_attr("fold_rmses", fold_rmses)
    return avg_rmse

# ======= Final train on Train+Val, test on Test =======
def final_fit_and_test(df, best_params, max_epochs, es_patience, outdir: Path):
    dfe = add_engineered_features(df)
    if best_params["log_target"]:
        dfe["_y"] = np.log1p(dfe[TARGET_COL].clip(lower=0)).astype(np.float32)
    else:
        dfe["_y"] = dfe[TARGET_COL].astype(np.float32)
    dfe = dfe.dropna().reset_index(drop=True)

    feat_cols = build_feat_list()
    X_all = dfe[feat_cols].to_numpy(np.float32)
    y_all = dfe[["_y"]].to_numpy(np.float32)

    n_total = len(dfe)
    n_test  = int(len(dfe) * (1 - (TRAIN_RATIO + VAL_RATIO)))
    n_tv    = n_total - n_test

    X_trval, y_trval = X_all[:n_tv], y_all[:n_tv]
    X_test,  y_test  = X_all[n_tv:], y_all[n_tv:]

    # scalers on Train+Val
    xsc = StandardScaler().fit(X_trval)
    ysc = StandardScaler().fit(y_trval)
    Xtrv_s, ytrv_s = xsc.transform(X_trval), ysc.transform(y_trval)
    Xte_s,  yte_s  = xsc.transform(X_test),  ysc.transform(y_test)

    # sequences
    lookback = best_params["lookback"]
    Xtrv_seq, ytrv_seq = make_sequences(Xtrv_s, ytrv_s, lookback)
    Xte_seq,  yte_seq  = make_sequences(Xte_s,  yte_s,  lookback)

    tr_loader = DataLoader(SeqDS(Xtrv_seq, ytrv_seq), batch_size=best_params["batch"], shuffle=True)
    te_loader = DataLoader(SeqDS(Xte_seq,  yte_seq ), batch_size=best_params["batch"], shuffle=False)

    model = BiLSTMRegressor(
        input_size=Xtrv_seq.shape[-1],
        hidden_size=best_params["hidden"],
        num_layers=best_params["layers"],
        dropout=best_params["dropout"],
        bidirectional=best_params["bidir"]
    ).to(DEVICE)

    loss_fn = nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=best_params["lr"], weight_decay=best_params["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt, T_0=10, T_mult=2, eta_min=1e-5
    )

    best = float("inf")
    no_improve = 0
    best_state = None

    for epoch in range(1, max_epochs + 1):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            if best_params["clip"] is not None:
                nn.utils.clip_grad_norm_(model.parameters(), best_params["clip"])
            opt.step()
        sched.step(epoch - 1)

        # (Optional) you can add quick ES using a held-out slice here
        # For now, full-epoch training on Train+Val.

    # Load best_state if you track it here; currently we just keep last epoch weights.
    # Evaluate on TEST in original units
    def inv_target(y_scaled):
        ys = ysc.inverse_transform(y_scaled).ravel()
        if best_params["log_target"]:
            ys = np.expm1(ys)
        return ys

    model.eval()
    preds_s, trues_s = [], []
    with torch.no_grad():
        for xb, yb in te_loader:
            pr = model(xb.to(DEVICE)).cpu().numpy()
            preds_s.append(pr)
            trues_s.append(yb.numpy())
    preds_s = np.vstack(preds_s)
    trues_s = np.vstack(trues_s)
    y_pred = inv_target(preds_s)
    y_true = inv_target(trues_s)

    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    smp  = smape(y_true, y_pred)

    # Save artifacts
    bilstm_dir = outdir / "biLSTM"
    bilstm_dir.mkdir(parents=True, exist_ok=True)

    torch.save(model.state_dict(), bilstm_dir / "bilstm_optuna_best.pt")
    joblib.dump(xsc, bilstm_dir / "x_scaler_optuna.pkl")
    joblib.dump(ysc, bilstm_dir / "y_scaler_optuna.pkl")
    with open(outdir / "best_params.json", "w") as f:
        json.dump(best_params, f, indent=2)

    plt.figure(figsize=(12, 5))
    plt.plot(y_true, label="Actual")
    plt.plot(y_pred, label="Pred (best Optuna)")
    plt.title("BiLSTM + FE — Test (Optuna best)")
    plt.xlabel("Test time steps")
    plt.ylabel(TARGET_COL)
    plt.legend()
    plt.tight_layout()
    plt.savefig(bilstm_dir / "test_plot_optuna.png", dpi=150)

    print("\n==== TEST (Optuna best) ====")
    print(f"RMSE : {rmse:.6f}")
    print(f"MAE  : {mae:.6f}")
    print(f"R^2  : {r2:.6f}")
    print(f"sMAPE: {smp:.2f}%")
    print(f"Saved to: {bilstm_dir.resolve()}")

# ======= CLI =======

file = "../WindPowerForecastingData.xlsx"
trials = 30
timeout = None
epochs = 80
patience = 12
seed = 42
outdir = "."




set_seed(seed)
outdir = Path(outdir)
outdir.mkdir(parents=True, exist_ok=True)

df = load_data(file)

study = optuna.create_study(direction="minimize")
study.optimize(
    lambda tr: objective(
        tr,
        df=df,
        max_epochs=min(30, epochs),   # e.g. 30 epochs for tuning
        es_patience=min(6, patience)  # smaller patience during tuning
    ),
    n_trials=trials,
    timeout=timeout,
    show_progress_bar=True,
    gc_after_trial=True
)


# Optuna results
print("\n==== OPTUNA BEST ====")
print("Value (avg Val RMSE):", study.best_value)
print("Params:", study.best_params)

# Coerce best params into a clean dict for final training
best_params = {
"lookback":   study.best_params["lookback"],
"hidden":     study.best_params["hidden"],
"layers":     study.best_params["layers"],
"dropout":    float(study.best_params["dropout"]),
"bidir": bool(study.best_params["bidir"]),
"batch":      study.best_params["batch"],
"lr":         float(study.best_params["lr"]),
"wd":         float(study.best_params["weight_decay"]),
"clip":       study.best_params["clip_norm"],
"log_target": bool(study.best_params["log_target"]),
}


# Final train on Train+Val, test on Test
final_fit_and_test(df, best_params, epochs, patience, outdir)


/tmp/ipykernel_6271/73529128.py:107: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df[TIME_COL] = pd.to_datetime(df[TIME_COL], infer_datetime_format=True, errors="coerce")
[I 2025-12-11 19:25:20,291] A new study created in memory with name: no-name-4848c775-0eff-4ebc-8c3f-d89fbaf62994
  0%|          | 0/30 [01:20<?, ?it/s]

[I 2025-12-11 19:26:40,899] Trial 0 finished with value: 0.10780308015416988 and parameters: {'lookback': 24, 'hidden': 256, 'layers': 1, 'dropout': 0.1, 'bidir': False, 'batch': 128, 'lr': 0.0003, 'weight_decay': 0.0001, 'clip_norm': 0.5, 'log_target': True}. Best is trial 0 with value: 0.10780308015416988.


Best trial: 0. Best value: 0.107803:   3%|▎         | 1/30 [02:44<39:05, 80.87s/it]

[W 2025-12-11 19:28:05,201] Trial 1 failed with parameters: {'lookback': 6, 'hidden': 256, 'layers': 1, 'dropout': 0.1, 'bidir': True, 'batch': 64, 'lr': 0.0007, 'weight_decay': 1e-05, 'clip_norm': 0.5, 'log_target': True} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/sulith/LSTM/LSTM_Wind_assignment/LSTM/lib/python3.12/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_6271/73529128.py", line 472, in <lambda>
    lambda tr: objective(
               ^^^^^^^^^^
  File "/tmp/ipykernel_6271/73529128.py", line 317, in objective
    rmse, _, _ = train_fold(
                 ^^^^^^^^^^^
  File "/tmp/ipykernel_6271/73529128.py", line 227, in train_fold
    pred = model(xb)
           ^^^^^^^^^
  File "/home/sulith/LSTM/LSTM_Wind_assignment/LSTM/lib/python3.12/site-packages/torch/nn/modules/module.py", line 1773, in _wrapped_call_imp

Best trial: 0. Best value: 0.107803:   3%|▎         | 1/30 [02:45<1:19:48, 165.13s/it]


KeyboardInterrupt: 

In [6]:
# tune_optuna_attention_wind.py
import argparse, json, math, random, copy
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import matplotlib.pyplot as plt

# ======= Columns =======
TIME_COL   = "TIMESTAMP"
TARGET_COL = "TARGETVAR"
BASE_FEATS = ["U10", "V10", "U100", "V100"]

# ======= Fixed FE =======
# User request: Keep lookbacks short and practical
LAGS_Y     = [1, 2, 3, 6, 12, 24] 
LAGS_SPEED = [1, 3, 6]
ROLLS_Y    = [6, 12, 24]

# ======= Splits =======
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ======= Repro =======
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# ======= FE =======
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create new features. Added Cubic Speed for physics correlation.
    """
    out = df.copy()
    # wind speed & direction
    out["speed10"]  = np.sqrt(out["U10"]**2  + out["V10"]**2)
    out["speed100"] = np.sqrt(out["U100"]**2 + out["V100"]**2)
    
    # === NEW: Physics feature (Power is proportional to v^3) ===
    out["speed100_cubed"] = out["speed100"] ** 3
    
    dir10  = np.arctan2(out["V10"],  out["U10"])
    dir100 = np.arctan2(out["V100"], out["U100"])
    out["dir10_sin"], out["dir10_cos"]   = np.sin(dir10),  np.cos(dir10)
    out["dir100_sin"], out["dir100_cos"] = np.sin(dir100), np.cos(dir100)

    # shear & veer
    out["shear_speed"] = out["speed100"] - out["speed10"]
    veer = dir100 - dir10
    out["veer_sin"], out["veer_cos"] = np.sin(veer), np.cos(veer)

    # time features
    out["hour"] = out[TIME_COL].dt.hour
    out["day"]  = out[TIME_COL].dt.dayofyear
    out["hour_sin"] = np.sin(2*np.pi*out["hour"]/24.0)
    out["hour_cos"] = np.cos(2*np.pi*out["hour"]/24.0)
    out["day_sin"]  = np.sin(2*np.pi*out["day"]/366.0)
    out["day_cos"]  = np.cos(2*np.pi*out["day"]/366.0)

    # target lags (shifted)
    for L in LAGS_Y:
        out[f"y_lag{L}"] = out[TARGET_COL].shift(L)

    # rolling means of y (shift to avoid leakage)
    for W in ROLLS_Y:
        out[f"y_roll{W}"] = (
            out[TARGET_COL].shift(1).rolling(W, min_periods=W).mean()
        )

    # speed lags
    for L in LAGS_SPEED:
        out[f"speed10_lag{L}"]  = out["speed10"].shift(L)
        out[f"speed100_lag{L}"] = out["speed100"].shift(L)

    return out


def build_feat_list():
    return (
        BASE_FEATS +
        [
            "speed10", "speed100", "speed100_cubed", # Added cubed
            "dir10_sin", "dir10_cos",
            "dir100_sin", "dir100_cos",
            "shear_speed", "veer_sin", "veer_cos",
            "hour_sin", "hour_cos",
            "day_sin", "day_cos"
        ] +
        [f"y_lag{L}" for L in LAGS_Y] +
        [f"y_roll{W}" for W in ROLLS_Y] +
        [f"speed10_lag{L}"  for L in LAGS_SPEED] +
        [f"speed100_lag{L}" for L in LAGS_SPEED]
    )

# ======= Data I/O =======
def load_data(file_path: str) -> pd.DataFrame:
    df = pd.read_excel(file_path)
    # Fixed deprecated argument warning
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.sort_values(TIME_COL).reset_index(drop=True)
    return df

def smape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return 100.0 * np.mean(
        2.0 * np.abs(y_pred - y_true) /
        (np.abs(y_pred) + np.abs(y_true) + eps)
    )

# ======= Sequences =======
def make_sequences(X, y, lookback):
    Xs, ys = [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i-lookback:i, :])
        ys.append(y[i, 0])
    return np.array(Xs, np.float32), np.array(ys, np.float32).reshape(-1, 1)

class SeqDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

# ======= Model: Attention BiLSTM =======
class AttentionBlock(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        # x: (Batch, Seq, Hidden)
        weights = torch.tanh(self.attn(x)) # (B, S, 1)
        weights = F.softmax(weights, dim=1) 
        # Weighted sum
        context = torch.sum(x * weights, dim=1) # (B, Hidden)
        return context, weights

class AttnBiLSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout, bidirectional=True):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional
        )
        self.rnn_out_dim = hidden_size * (2 if bidirectional else 1)
        
        # Attention Layer
        self.attention = AttentionBlock(self.rnn_out_dim)
        
        self.norm = nn.LayerNorm(self.rnn_out_dim)
        
        # Head
        self.head = nn.Sequential(
            nn.Linear(self.rnn_out_dim, self.rnn_out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.rnn_out_dim, 1),
        )

    def forward(self, x):
        # x: (Batch, Seq_len, Input_size)
        o, _ = self.lstm(x)  # (B, T, H_out)
        
        # Apply Attention
        context, _ = self.attention(o) # (B, H_out)
        
        context = self.norm(context)
        return self.head(context)

# ======= Training =======
def train_fold(Xtr, ytr, Xva, yva, params, max_epochs, es_patience, log_target, clip_norm):
    # Using RobustScaler to handle outliers better
    xsc = RobustScaler().fit(Xtr)
    ysc = StandardScaler().fit(ytr)
    
    Xtr_s, ytr_s = xsc.transform(Xtr), ysc.transform(ytr)
    Xva_s, yva_s = xsc.transform(Xva), ysc.transform(yva)

    lookback = params["lookback"]
    Xtr_seq, ytr_seq = make_sequences(Xtr_s, ytr_s, lookback)
    Xva_seq, yva_seq = make_sequences(Xva_s, yva_s, lookback)

    if len(Xtr_seq) < 16 or len(Xva_seq) < 16:
        return float("inf"), None, None

    tr_loader = DataLoader(SeqDS(Xtr_seq, ytr_seq), batch_size=params["batch"], shuffle=True, drop_last=True)
    va_loader = DataLoader(SeqDS(Xva_seq, yva_seq), batch_size=params["batch"], shuffle=False)

    model = AttnBiLSTMRegressor(
        input_size=Xtr_seq.shape[-1],
        hidden_size=params["hidden"],
        num_layers=params["layers"],
        dropout=params["dropout"],
        bidirectional=params["bidir"]
    ).to(DEVICE)

    # Huber Loss is more robust to wind power spikes/outliers than MSE
    loss_fn = nn.HuberLoss(delta=1.0) 
    
    opt = torch.optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["wd"])
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=params["lr"], 
        steps_per_epoch=len(tr_loader), 
        epochs=max_epochs,
        pct_start=0.3
    )

    def inv_target(y_scaled):
        y = ysc.inverse_transform(y_scaled).ravel()
        if log_target:
            y = np.expm1(y)
        return y

    best_rmse = float("inf")
    no_improve = 0
    best_state = None

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses = []
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            if clip_norm is not None:
                nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
            opt.step()
            sched.step()
            train_losses.append(loss.item())

        model.eval()
        preds_s, trues_s = [], []
        with torch.no_grad():
            for xb, yb in va_loader:
                pr = model(xb.to(DEVICE)).cpu().numpy()
                preds_s.append(pr)
                trues_s.append(yb.numpy())
        
        preds_s = np.vstack(preds_s)
        trues_s = np.vstack(trues_s)
        y_pred = inv_target(preds_s)
        y_true = inv_target(trues_s)
        
        # Calculate RMSE
        rmse = math.sqrt(mean_squared_error(y_true, y_pred))

        if rmse < best_rmse:
            best_rmse = rmse
            no_improve = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            no_improve += 1
            if no_improve >= es_patience:
                break
                
    return best_rmse, best_state, (xsc, ysc)

def make_walkforward_indices(n_total, n_test):
    n_tv = n_total - n_test
    # 3 Folds
    cut1 = int(n_tv * 0.60); val1 = int(n_tv * 0.75)
    cut2 = int(n_tv * 0.75); val2 = int(n_tv * 0.85)
    cut3 = int(n_tv * 0.85); val3 = int(n_tv * 0.95)

    folds = [
        (0, cut1, cut1, val1),
        (0, cut2, cut2, val2),
        (0, cut3, cut3, val3),
    ]
    return folds

# ======= Objective =======
def objective(trial, df: pd.DataFrame, max_epochs: int, es_patience: int):
    params = {
        "lookback": trial.suggest_categorical("lookback", [6, 12, 24]),
        "hidden":   trial.suggest_categorical("hidden",   [64, 128, 256]),
        "layers":   trial.suggest_int("layers", 1, 2),
        "dropout":  trial.suggest_float("dropout", 0.1, 0.5),
        "bidir":    trial.suggest_categorical("bidir", [True, False]),
        "batch":    trial.suggest_categorical("batch", [64, 128]),
        "lr":       trial.suggest_float("lr", 1e-4, 5e-3, log=True),
        "wd":       trial.suggest_float("wd", 1e-6, 1e-3, log=True),
        "clip":     trial.suggest_categorical("clip_norm", [1.0]),
        "log_target": trial.suggest_categorical("log_target", [False, True]),
    }

    dfe = add_engineered_features(df)
    if params["log_target"]:
        dfe["_y"] = np.log1p(dfe[TARGET_COL].clip(lower=0)).astype(np.float32)
    else:
        dfe["_y"] = dfe[TARGET_COL].astype(np.float32)
    dfe = dfe.dropna().reset_index(drop=True)

    feat_cols = build_feat_list()
    X_all = dfe[feat_cols].to_numpy(np.float32)
    y_all = dfe[["_y"]].to_numpy(np.float32)

    n_total = len(dfe)
    n_test  = int(len(dfe) * (1 - (TRAIN_RATIO + VAL_RATIO)))
    folds   = make_walkforward_indices(n_total, n_test)

    fold_rmses = []
    # CRITICAL CHANGE: We iterate over ALL folds, not just the last one.
    # This prevents overfitting to a specific month.
    for i, (tr_start, tr_end, va_start, va_end) in enumerate(folds):
        Xtr = X_all[tr_start:tr_end]; ytr = y_all[tr_start:tr_end]
        Xva = X_all[va_start:va_end]; yva = y_all[va_start:va_end]
        
        rmse, _, _ = train_fold(
            Xtr, ytr, Xva, yva, params, max_epochs, es_patience,
            log_target=params["log_target"], clip_norm=params["clip"]
        )
        fold_rmses.append(rmse)
        
        # Pruning (Optional: stop bad trials early based on first fold)
        trial.report(rmse, i)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(fold_rmses)

# ======= Final Fit with ENSEMBLING =======
def final_fit_and_test(df, best_params, max_epochs, es_patience, outdir: Path, n_seeds=5):
    """
    Trains n_seeds models with the best params and averages their predictions.
    This reduces variance and usually improves RMSE significantly.
    """
    dfe = add_engineered_features(df)
    if best_params["log_target"]:
        dfe["_y"] = np.log1p(dfe[TARGET_COL].clip(lower=0)).astype(np.float32)
    else:
        dfe["_y"] = dfe[TARGET_COL].astype(np.float32)
    dfe = dfe.dropna().reset_index(drop=True)

    feat_cols = build_feat_list()
    X_all = dfe[feat_cols].to_numpy(np.float32)
    y_all = dfe[["_y"]].to_numpy(np.float32)

    n_total = len(dfe)
    n_test  = int(len(dfe) * (1 - (TRAIN_RATIO + VAL_RATIO)))
    n_tv    = n_total - n_test

    X_trval, y_trval = X_all[:n_tv], y_all[:n_tv]
    X_test,  y_test  = X_all[n_tv:], y_all[n_tv:]

    # Use a Hold-out validation set from TrVal for Early Stopping
    val_len = int(len(X_trval) * 0.1)
    train_len = len(X_trval) - val_len
    
    # Pre-split for training to allow proper early stopping
    X_t, y_t = X_trval[:train_len], y_trval[:train_len]
    X_v, y_v = X_trval[train_len:], y_trval[train_len:]

    ensemble_preds = []
    
    print(f"\nTraining Ensemble of {n_seeds} models...")
    
    bilstm_dir = outdir / "AttnBiLSTM"
    bilstm_dir.mkdir(parents=True, exist_ok=True)

    # LOOP FOR ENSEMBLING
    for seed_i in range(n_seeds):
        current_seed = 42 + seed_i
        set_seed(current_seed)
        print(f"  > Model {seed_i+1}/{n_seeds} (Seed {current_seed})")

        # Train with early stopping using the internal split
        rmse, state_dict, (xsc, ysc) = train_fold(
            X_t, y_t, X_v, y_v, 
            best_params, max_epochs, es_patience, 
            log_target=best_params["log_target"], 
            clip_norm=best_params["clip"]
        )

        # Re-build model structure for inference
        model = AttnBiLSTMRegressor(
            input_size=X_test.shape[-1], # placeholder dim, will match scaler
            hidden_size=best_params["hidden"],
            num_layers=best_params["layers"],
            dropout=best_params["dropout"],
            bidirectional=best_params["bidir"]
        ).to(DEVICE)
        
        model.load_state_dict(state_dict)
        model.eval()

        # Inference on Test
        Xte_s = xsc.transform(X_test)
        Xte_seq, _ = make_sequences(Xte_s, np.zeros_like(y_test), best_params["lookback"])
        
        # Test loader
        te_ds = SeqDS(Xte_seq, np.zeros((len(Xte_seq), 1)))
        te_loader = DataLoader(te_ds, batch_size=best_params["batch"], shuffle=False)
        
        fold_preds = []
        with torch.no_grad():
            for xb, _ in te_loader:
                pr = model(xb.to(DEVICE)).cpu().numpy()
                fold_preds.append(pr)
        fold_preds = np.vstack(fold_preds)
        
        # Invert scaling
        y_pred_fold = ysc.inverse_transform(fold_preds).ravel()
        if best_params["log_target"]:
            y_pred_fold = np.expm1(y_pred_fold)
            
        ensemble_preds.append(y_pred_fold)
        
        # Save individual model
        torch.save(state_dict, bilstm_dir / f"model_seed_new_{current_seed}.pt")

    # AVERAGE PREDICTIONS
    avg_preds = np.mean(ensemble_preds, axis=0)
    
    # Align Truth (slice off lookback)
    lookback = best_params["lookback"]
    y_true_aligned = y_test[lookback:] 
    if best_params["log_target"]:
        y_true_aligned = np.expm1(y_true_aligned)
    
    y_true_aligned = y_true_aligned.ravel()

    # Metrics
    rmse = math.sqrt(mean_squared_error(y_true_aligned, avg_preds))
    mae  = mean_absolute_error(y_true_aligned, avg_preds)
    r2   = r2_score(y_true_aligned, avg_preds)
    smp  = smape(y_true_aligned, avg_preds)

    # Save Results
    joblib.dump(xsc, bilstm_dir / "x_scaler_new.pkl") # saving last scaler
    joblib.dump(ysc, bilstm_dir / "y_scaler_new.pkl")
    with open(outdir / "best_params_new.json", "w") as f:
        json.dump(best_params, f, indent=2)

    plt.figure(figsize=(12, 5))
    plt.plot(y_true_aligned, label="Actual", alpha=0.7)
    plt.plot(avg_preds, label=f"Ensemble Pred ({n_seeds} models)", alpha=0.8)
    plt.title("Attention BiLSTM Ensemble — Test Set")
    plt.xlabel("Test time steps")
    plt.ylabel(TARGET_COL)
    plt.legend()
    plt.tight_layout()
    plt.savefig(bilstm_dir / "test_plot_ensemble_new.png", dpi=150)

    print("\n==== TEST (Ensemble Result) ====")
    print(f"RMSE : {rmse:.6f}")
    print(f"MAE  : {mae:.6f}")
    print(f"R^2  : {r2:.6f}")
    print(f"sMAPE: {smp:.2f}%")
    print(f"Saved to: {bilstm_dir.resolve()}")

# ======= CLI =======
file = "../WindPowerForecastingData.xlsx"
trials = 20       
timeout = None
epochs = 80
patience = 12
seed = 42
outdir = "."

set_seed(seed)
outdir = Path(outdir)
outdir.mkdir(parents=True, exist_ok=True)

df = load_data(file)

study = optuna.create_study(direction="minimize")
study.optimize(
    lambda tr: objective(
        tr,
        df=df,
        max_epochs=min(35, epochs), 
        es_patience=min(8, patience) 
    ),
    n_trials=trials,
    timeout=timeout,
    show_progress_bar=True,
    gc_after_trial=True
)


print("\n==== OPTUNA BEST ====")
print("Value (avg Val RMSE):", study.best_value)
print("Params:", study.best_params)

best_params = {
    "lookback":   study.best_params["lookback"],
    "hidden":     study.best_params["hidden"],
    "layers":     study.best_params["layers"],
    "dropout":    float(study.best_params["dropout"]),
    "bidir":      bool(study.best_params["bidir"]),
    "batch":      study.best_params["batch"],
    "lr":         float(study.best_params["lr"]),
    "wd":         float(study.best_params["wd"]),
    "clip":       study.best_params["clip_norm"],
    "log_target": bool(study.best_params["log_target"]),
}

# Final train with 5-seed Ensemble
final_fit_and_test(df, best_params, epochs, patience, outdir, n_seeds=5)

[I 2025-12-11 19:30:45,393] A new study created in memory with name: no-name-6326ed9f-85bd-4c38-a6ba-dff102266d80
  0%|          | 0/20 [02:04<?, ?it/s]

[I 2025-12-11 19:32:49,511] Trial 0 finished with value: 0.12873451519036963 and parameters: {'lookback': 12, 'hidden': 64, 'layers': 1, 'dropout': 0.11602487532019344, 'bidir': False, 'batch': 64, 'lr': 0.00024406873460964764, 'wd': 1.631854552491279e-05, 'clip_norm': 1.0, 'log_target': True}. Best is trial 0 with value: 0.12873451519036963.


Best trial: 0. Best value: 0.128735:   5%|▌         | 1/20 [15:23<39:22, 124.36s/it]

[W 2025-12-11 19:46:08,900] Trial 1 failed with parameters: {'lookback': 24, 'hidden': 256, 'layers': 1, 'dropout': 0.2911256777775693, 'bidir': True, 'batch': 64, 'lr': 0.0019676053129522777, 'wd': 0.00011072966926114004, 'clip_norm': 1.0, 'log_target': False} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/sulith/LSTM/LSTM_Wind_assignment/LSTM/lib/python3.12/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_6271/4070857494.py", line 489, in <lambda>
    lambda tr: objective(
               ^^^^^^^^^^
  File "/tmp/ipykernel_6271/4070857494.py", line 329, in objective
    rmse, _, _ = train_fold(
                 ^^^^^^^^^^^
  File "/tmp/ipykernel_6271/4070857494.py", line 244, in train_fold
    loss.backward()
  File "/home/sulith/LSTM/LSTM_Wind_assignment/LSTM/lib/python3.12/site-packages/torch/_tensor.py", line 647, in backw

Best trial: 0. Best value: 0.128735:   5%|▌         | 1/20 [15:23<4:52:32, 923.80s/it]


KeyboardInterrupt: 